# Spark Structured Streaming — Micro-batch, Watermarking, State

## Mental model

Spark Structured Streaming treats an unbounded stream as a continuously updated table.  
Kafka events land in that table, Spark processes them in **micro-batches**, and stateful operators
like windows and aggregations maintain evolving state across batches.

For this Citi-flavored demo:

- **Source**: Kafka topic `citi.stream.spark`
- **Engine**: Spark Structured Streaming in `local[*]`
- **Pattern**: watermarking + stateful aggregation + checkpointing
- **Why it matters**: this is the practical path from batch ETL teams into streaming without forcing a full rewrite into a different programming model

### Environment context

- PostgreSQL: `localhost:5432`, database `de_telemetry`, user `de_admin`
- Tables:
  - `endpoints` — 10,000 rows
  - `metrics` — 500,000 rows
  - `alerts` — 25,000 rows
- Kafka: `localhost:9092`
- Spark: `pyspark==3.5.4`
- Citi narrative: 6,000+ API endpoints monitored for latency, error rate, throughput, and alert escalation tiers


In [ ]:
import json
import os
import time
import uuid
from datetime import datetime, timedelta, timezone

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# -----------------------------------------------------------------------------
# Environment / stack configuration from the provided context
# -----------------------------------------------------------------------------
os.environ["JAVA_HOME"] = r"C:/Program Files/Java/jre1.8.0_481"
os.environ["HADOOP_HOME"] = r"C:/hadoop"

KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC = "citi.stream.spark"
CHECKPOINT_ROOT = os.path.abspath("./checkpoints/spark_structured_streaming")
ALERT_COUNTS_CHECKPOINT = os.path.join(CHECKPOINT_ROOT, "alert_counts")

POSTGRES_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

DATASET_CONTEXT = {
    "endpoints_rows": 10_000,
    "metrics_rows": 500_000,
    "alerts_rows": 25_000,
    "narrative": "6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers",
}

os.makedirs(ALERT_COUNTS_CHECKPOINT, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("spark_structured_streaming_kafka_demo")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark app name:", spark.sparkContext.appName)
print("Kafka bootstrap:", KAFKA_BOOTSTRAP)
print("Checkpoint location:", ALERT_COUNTS_CHECKPOINT)
print("PostgreSQL config:", {k: v for k, v in POSTGRES_CONFIG.items() if k != "password"})


## Produce stream into Kafka

This cell creates the Kafka topic if needed and pushes **100 alert events** into
`citi.stream.spark` using `confluent-kafka`.

A few records are intentionally made late by more than 30 seconds so the watermark behavior
becomes visible in the downstream aggregation.


In [ ]:
admin = AdminClient({"bootstrap.servers": KAFKA_BOOTSTRAP})

existing_topics = admin.list_topics(timeout=10).topics
if KAFKA_TOPIC not in existing_topics:
    futures = admin.create_topics(
        [NewTopic(KAFKA_TOPIC, num_partitions=1, replication_factor=1)]
    )
    for topic_name, future in futures.items():
        try:
            future.result()
            print(f"Created topic: {topic_name}")
        except Exception as exc:
            # Safe if the topic was created concurrently
            print(f"Topic create note for {topic_name}: {exc}")
else:
    print(f"Topic already exists: {KAFKA_TOPIC}")

producer = Producer({"bootstrap.servers": KAFKA_BOOTSTRAP})

delivery_results = {"delivered": 0, "failed": 0}

def delivery_report(err, msg):
    if err is not None:
        delivery_results["failed"] += 1
        print(f"Delivery failed for key={msg.key()}: {err}")
    else:
        delivery_results["delivered"] += 1

severities = ["info", "warning", "critical"]
regions = ["us-east-1", "us-west-2", "eu-west-1"]
metric_names = ["latency_ms", "error_rate", "throughput_rps"]

base_time = datetime.now(timezone.utc)

for i in range(100):
    severity = severities[i % len(severities)]
    region = regions[i % len(regions)]
    metric_name = metric_names[i % len(metric_names)]

    # Most records are current; some are intentionally late.
    if i % 15 == 0:
        event_time = base_time - timedelta(seconds=45)   # likely dropped by watermark later
    elif i % 10 == 0:
        event_time = base_time - timedelta(seconds=15)   # late but within watermark
    else:
        event_time = base_time + timedelta(seconds=i % 7)

    payload = {
        "event_id": str(uuid.uuid4()),
        "alert_id": 100000 + i,
        "endpoint_id": 1000 + (i % 250),
        "endpoint_name": f"api-endpoint-{i % 250:03d}",
        "region": region,
        "severity": severity,
        "metric_name": metric_name,
        "metric_value": float(20 + (i % 11) * 3.5),
        "message": f"{severity.upper()} alert for endpoint {1000 + (i % 250)}",
        "created_at": event_time.isoformat(),
    }

    producer.produce(
        KAFKA_TOPIC,
        key=str(payload["endpoint_id"]),
        value=json.dumps(payload).encode("utf-8"),
        callback=delivery_report,
    )
    producer.poll(0)

producer.flush(15)

print("Delivery results:", delivery_results)
assert delivery_results["delivered"] == 100, f"Expected 100 delivered messages, got {delivery_results['delivered']}"


## Read from Kafka and parse the JSON payload

Structured Streaming reads Kafka records as bytes. We cast the payload to string,
apply a schema, and extract a typed event stream with a proper event timestamp.


In [ ]:
alert_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("alert_id", IntegerType(), False),
    StructField("endpoint_id", IntegerType(), False),
    StructField("endpoint_name", StringType(), False),
    StructField("region", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("metric_name", StringType(), False),
    StructField("metric_value", DoubleType(), False),
    StructField("message", StringType(), False),
    StructField("created_at", StringType(), False),
])

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream = (
    raw_stream
    .selectExpr("CAST(key AS STRING) AS kafka_key", "CAST(value AS STRING) AS json_payload", "timestamp AS kafka_timestamp")
    .select(
        "kafka_key",
        "kafka_timestamp",
        F.from_json(F.col("json_payload"), alert_schema).alias("payload")
    )
    .select(
        "kafka_key",
        "kafka_timestamp",
        "payload.*"
    )
    .withColumn("created_at", F.to_timestamp("created_at"))
)

print("Is streaming DataFrame:", parsed_stream.isStreaming)
parsed_stream.printSchema()


## Stateful count with watermark

We aggregate alerts by:

- **severity**
- **1-minute event-time window**
- **watermark = 30 seconds**

Why watermarking exists:

- Spark keeps state for windowed aggregations
- without a watermark, old state can grow forever
- with a watermark, Spark knows when it can safely evict old window state


In [ ]:
# Clean up an old in-session memory sink query if this notebook cell is re-run.
for q in spark.streams.active:
    if q.name == "alert_counts":
        q.stop()
        q.awaitTermination(10)

windowed_counts = (
    parsed_stream
    .withWatermark("created_at", "30 seconds")
    .groupBy(
        F.window("created_at", "1 minute"),
        F.col("severity")
    )
    .count()
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("severity"),
        F.col("count")
    )
)

alert_counts_query = (
    windowed_counts.writeStream
    .queryName("alert_counts")
    .format("memory")
    .outputMode("complete")
    .option("checkpointLocation", ALERT_COUNTS_CHECKPOINT)
    .trigger(processingTime="5 seconds")
    .start()
)

time.sleep(10)
alert_counts_query.processAllAvailable()

spark.sql("""
    SELECT
        window_start,
        window_end,
        severity,
        count
    FROM alert_counts
    ORDER BY window_start, severity
""").show(100, truncate=False)


## Exactly-once with checkpointing

Spark Structured Streaming relies on **checkpointing** to recover offsets and state.

For Kafka + stateful aggregation, the checkpoint holds:

- source offsets already committed into query progress
- state store metadata for the aggregation
- progress logs that let Spark resume after failure

That is the practical basis for end-to-end exactly-once style behavior in this design:
Spark can restart, reload state, and continue from the correct Kafka offsets instead of replaying blindly.


In [ ]:
checkpoint_config = {
    "checkpoint_root": CHECKPOINT_ROOT,
    "alert_counts_checkpoint": ALERT_COUNTS_CHECKPOINT,
    "active_query_name": alert_counts_query.name,
    "active_query_id": str(alert_counts_query.id),
    "active_run_id": str(alert_counts_query.runId),
    "spark_sql_shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "spark_session_timezone": spark.conf.get("spark.sql.session.timeZone"),
    "spark_kafka_package": spark.conf.get("spark.jars.packages"),
}

for key, value in checkpoint_config.items():
    print(f"{key}: {value}")


## Output modes — append vs complete vs update

**Append**
- only newly finalized rows are emitted
- best when windows are closed and no more updates are expected

```python
append_query = (
    windowed_counts.writeStream
    .format("console")
    .outputMode("append")
    .option("checkpointLocation", "./checkpoints/append")
    .start()
)
```

**Complete**
- the full result table is emitted every trigger
- useful for demos and memory sinks

```python
complete_query = (
    windowed_counts.writeStream
    .format("memory")
    .queryName("alert_counts")
    .outputMode("complete")
    .option("checkpointLocation", "./checkpoints/complete")
    .start()
)
```

**Update**
- only rows changed since the last trigger are emitted
- great for incremental dashboards and downstream consumers

```python
update_query = (
    windowed_counts.writeStream
    .format("console")
    .outputMode("update")
    .option("checkpointLocation", "./checkpoints/update")
    .start()
)
```


In [ ]:
output_mode_examples = {
    "append": {
        "best_for": "finalized rows only",
        "typical_use": "closed windows, downstream append sinks",
    },
    "complete": {
        "best_for": "entire aggregate table each trigger",
        "typical_use": "memory sink demos, small aggregate views",
    },
    "update": {
        "best_for": "only changed rows since last trigger",
        "typical_use": "incremental dashboards and streaming consumers",
    },
}

for mode, details in output_mode_examples.items():
    print(f"{mode.upper()}: {details}")


## What just happened

You pushed finite alert events into Kafka, then let Spark Structured Streaming treat that topic as a streaming table.

### Compared with Kafka Streams
- Kafka Streams is library-first and tightly centered on Kafka
- Spark is DataFrame-first and often easier for batch ETL teams already living in Spark SQL / PySpark
- Kafka Streams usually feels more application-embedded; Spark feels more analytics-engine centered

### Compared with Flink
- Flink is more native-streaming in posture
- Spark Structured Streaming is micro-batch oriented but operationally familiar for Spark teams
- Flink often wins when ultra-low-latency event-at-a-time processing is the priority

### Citi tie-in
**Spark Structured Streaming on Kafka is Citi's go-to for batch-to-streaming migration — same DataFrame API, same team, different trigger.**


In [ ]:
print("Active queries before cleanup:", [q.name for q in spark.streams.active])

for q in spark.streams.active:
    q.stop()
    q.awaitTermination(10)

print("Active queries after cleanup:", [q.name for q in spark.streams.active])

spark.stop()
print("Spark session stopped cleanly.")
